# TrOCR run 4: longer mixed fine-tune

Same recipe as run 3 (synthetic print + real EHRI typewritten Polish lines),
but **15 epochs** instead of 5. run 3's val CER was still falling monotonically
at epoch 5, so the model had not reached plateau. Best checkpoint is selected by
val CER, so late-epoch overfitting is automatically discarded.

Starts fresh from `PiotrSty/trocr-pl-base` (not from run 3) to avoid cascading
overfit on the small EHRI set. Evaluates run 4 against run 3 (`trocr-pl-mixed-v1`)
and run 2 (`trocr-pl-base`) on the frozen EHRI test docs and the printed
`real-lines-v1` benchmark. Select GPU T4. All revisions pinned.

In [ ]:
import subprocess, sys, os
from pathlib import Path
CODE_REVISION = '5d2d06a971d051d21313bf1a11af575ed8a2885d'
BASE_MODEL = 'PiotrSty/trocr-pl-base'
BASE_REVISION = 'fff0416a9ccd8786cbd6f12d4cf3147c07056b18'
RUN3_MODEL = 'PiotrSty/trocr-pl-mixed-v1'
SYN_REVISION = 'd881debb90045fd71ad8e25faeeafeb6adab6622'
EHRI_REVISION = '3003e8614b74a351e7d94aba4f1348368815fb70'
repo = Path('/kaggle/working/OCR_engine')
if not repo.exists():
    subprocess.run(['git','clone','https://github.com/PiotrStyla/OCR_engine.git',str(repo)],check=True)
subprocess.run(['git','-C',str(repo),'fetch','origin',CODE_REVISION],check=True)
subprocess.run(['git','-C',str(repo),'checkout','--detach',CODE_REVISION],check=True)
os.chdir(repo)
sys.path.insert(0,str(repo))
subprocess.run([sys.executable,'-m','pip','uninstall','-y','torchao'],check=True)
subprocess.run([sys.executable,'-m','pip','install','transformers==4.57.6','peft==0.19.1','jiwer','pillow','accelerate'],check=True)

In [ ]:
import tarfile
import torch
assert torch.cuda.is_available(), 'GPU is required; select Kaggle GPU T4.'
print('GPU:', torch.cuda.get_device_name(0), 'torch:', torch.__version__)
probe = torch.ones((2, 2), device='cuda')
assert (probe @ probe).sum().item() == 8.0
torch.cuda.synchronize(); del probe
print('CUDA_PREFLIGHT_OK', flush=True)
from huggingface_hub import hf_hub_download
from training.protocol import pair_manifest

syn_archive = hf_hub_download('PiotrSty/ocr-pl-lines','ocr-pl-lines-v1.tar.gz',repo_type='dataset',revision=SYN_REVISION)
syn_root = Path('/kaggle/working/ocr-pl-lines-v1'); syn_root.mkdir(parents=True, exist_ok=True)
with tarfile.open(syn_archive,'r:gz') as b: b.extractall(syn_root, filter='data')

ehri_archive = hf_hub_download('PiotrSty/ehri-pl-lines','ehri-pl-lines-v1.tar.gz',repo_type='dataset',revision=EHRI_REVISION)
ehri_root = Path('/kaggle/working/ehri-pl-lines-v1'); ehri_root.mkdir(parents=True, exist_ok=True)
with tarfile.open(ehri_archive,'r:gz') as b: b.extractall(ehri_root, filter='data')

print('synthetic train:', len(pair_manifest(syn_root/'train')), 'val:', len(pair_manifest(syn_root/'val')))
print('ehri train:', len(pair_manifest(ehri_root/'train')), 'dev:', len(pair_manifest(ehri_root/'dev')), 'test:', len(pair_manifest(ehri_root/'test')))

In [ ]:
# Fine-tune run-2 on mixed synthetic + real EHRI lines; 15 epochs; validate on held-out EHRI dev.
output = '/kaggle/working/trocr-pl-run4'
subprocess.run([sys.executable,'-m','training.train_trocr_pl',
    '--train-dir',f'{syn_root}/train',f'{ehri_root}/train',
    '--val-dir',f'{ehri_root}/dev',
    '--base',BASE_MODEL,'--revision',BASE_REVISION,'--output',output,
    '--epochs','15','--batch-size','8','--lr','1e-4','--no-4bit'],check=True)
print(Path(output,'selection.json').read_text())
print(Path(output,'best_metrics.json').read_text())
# No upload_folder: review CER on held-out test before any promotion.

In [ ]:
# Final evaluation on frozen held-out sets — self-contained cell (safe to rerun alone).
# Compares run4 vs run3 (mixed-v1) vs run2-base on EHRI test (typewriter) and real-lines-v1 (print).
import sys, subprocess
from pathlib import Path
sys.path.insert(0, '/kaggle/working/OCR_engine')
BASE_MODEL = 'PiotrSty/trocr-pl-base'
RUN3_MODEL = 'PiotrSty/trocr-pl-mixed-v1'
output = '/kaggle/working/trocr-pl-run4'
ehri_root = Path('/kaggle/working/ehri-pl-lines-v1')
real_lines = '/kaggle/working/OCR_engine/benchmarks/real-lines-v1/pairs'
for name, model in [('run4', output), ('run3', RUN3_MODEL), ('run2-base', BASE_MODEL)]:
    for split, data in [('ehri-test', f'{ehri_root}/test'), ('real-lines-v1', real_lines)]:
        print(f'=== {name} on {split} ===', flush=True)
        subprocess.run([sys.executable,'-m','training.evaluate','--data',data,
            '--model',model,'--device','cuda','--batch-size','16'], check=True)

In [ ]:
# Publish run4 to Hugging Face as a SEPARATE experimental model (run4).
# Does NOT overwrite trocr-pl-base or trocr-pl-mixed-v1. Uses Kaggle Secret "HF_TOKEN".
import os, json
from pathlib import Path
from huggingface_hub import HfApi, create_repo

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
assert token, "Add a Kaggle Secret named HF_TOKEN with write scope."

api = HfApi(token=token)
repo_id = "PiotrSty/trocr-pl-mixed-v2"
create_repo(repo_id, repo_type="model", exist_ok=True, token=token)

model_dir = "/kaggle/working/trocr-pl-run4"
assert Path(model_dir, "model.safetensors").exists(), f"missing merged model in {model_dir}"

sel = json.loads(Path(model_dir, 'selection.json').read_text())
met = json.loads(Path(model_dir, 'best_metrics.json').read_text())
best_ckpt = sel.get('best_checkpoint', '?')
best_cer = met.get('eval_cer', float('nan'))
best_wer = met.get('eval_wer', float('nan'))

card = f'''---
language: [pl]
license: apache-2.0
base_model: PiotrSty/trocr-pl-base
tags: [trocr, ocr, polish, historical, typewriter, qlora, peft, mixed-data]
library_name: transformers
---

# PiotrSty/trocr-pl-mixed-v2 (experimental)

Longer fine-tune of **PiotrSty/trocr-pl-base** on synthetic Polish print +
real EHRI typewritten Polish lines (CC-BY 4.0, ehri-pl-lines). Same recipe as
trocr-pl-mixed-v1 (run 3) but 15 epochs instead of 5, because run 3's val CER
was still falling at epoch 5.

## Training

- Base: PiotrSty/trocr-pl-base
- Method: QLoRA on decoder attention (q/k/v/out_proj), rank 16, alpha 32
- Train: 2000 synthetic lines + 349 real EHRI lines (3 docs)
- Val: 38 EHRI lines (held-out doc ZIH3010905)
- Epochs: 15, batch 8, lr 1e-4, T4 x2
- Best checkpoint: {best_ckpt} (val CER {best_cer:.4f}, WER {best_wer:.4f})
- Document-level split, no line leakage.

## Evaluation on frozen held-out sets

Replace the numbers below with the actual run4 evaluation output from the
notebook before publishing, or rerun `training.evaluate` on this model after
download. Reference (run3 mixed-v1):

| Model | EHRI test (81, typewriter) | real-lines-v1 (75, print) |
|---|---:|---:|
| trocr-pl-base (run2) | CER 47.30% / WER 90.82% | CER 11.11% / WER 35.84% |
| trocr-pl-mixed-v1 (run3, 5 ep) | CER 33.95% / WER 85.69% | CER 7.09% / WER 29.44% |
| trocr-pl-mixed-v2 (run4, 15 ep) | (fill from eval cell) | (fill from eval cell) |

## Limitations

- Line recognizer only; page segmentation on faded typewriter is unreliable.
- Only 349 real typewritten training lines.
- Do NOT use as drop-in replacement without your own eval.

## Provenance

See run.json, selection.json, best_metrics.json in this repo.
Source: https://github.com/PiotrStyla/OCR_engine (commit 7065e6a)
EHRI dataset: https://huggingface.co/datasets/PiotrSty/ehri-pl-lines
'''
Path(model_dir, "README.md").write_text(card, encoding="utf-8")

api.upload_folder(
    folder_path=model_dir,
    repo_id=repo_id,
    repo_type="model",
    token=token,
    ignore_patterns=["checkpoint-*", "optimizer.pt", "scheduler.pt", "rng_state*.pth", "training_args.bin"],
)
print(f"Published: https://huggingface.co/{repo_id}")